In [ ]:
import os
import numpy as np
import pandas as pd
import pickle

from code_libraries.training_metrics import *
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score

# from code_libraries.xgb_trainter_lib import XGBoostGS
from code_libraries.logreg_momentum_lib import LogisticMomentumGS

## Paths

In [ ]:
main_dir = "../"
save_dir = os.path.join(main_dir, "heavy_data_NO_GIT/models_scalers_by_chain")
data_arrays=   os.path.join(main_dir, "heavy_data_NO_GIT/pkl_by_chain_logreg")
os.makedirs(save_dir, exist_ok=True)
os.makedirs(data_arrays, exist_ok=True)

# By-chain feature CSVs (sn1 and sn2 separately)
df_path_sn1_3pt = os.path.join(main_dir, "heavy_data_NO_GIT/dataframes_by_chain/sym/training_sn1_ft_3pt.csv")
df_path_sn2_3pt = os.path.join(main_dir, "heavy_data_NO_GIT/dataframes_by_chain/sym/training_sn2_ft_3pt.csv")

df_path_sn1_5pt = os.path.join(main_dir, "heavy_data_NO_GIT/dataframes_by_chain/sym/training_sn1_ft_5pt.csv")
df_path_sn2_5pt = os.path.join(main_dir, "heavy_data_NO_GIT/dataframes_by_chain/sym/training_sn2_ft_5pt.csv")

df_path_sn1_7pt = os.path.join(main_dir, "heavy_data_NO_GIT/dataframes_by_chain/sym/training_sn1_ft_7pt.csv")
df_path_sn2_7pt = os.path.join(main_dir, "heavy_data_NO_GIT/dataframes_by_chain/sym/training_sn2_ft_7pt.csv")

df_path_sn1_9pt = os.path.join(main_dir, "heavy_data_NO_GIT/dataframes_by_chain/sym/training_sn1_ft_9pt.csv")
df_path_sn2_9pt = os.path.join(main_dir, "heavy_data_NO_GIT/dataframes_by_chain/sym/training_sn2_ft_9pt.csv")

df_path_sn1_11pt = os.path.join(main_dir, "heavy_data_NO_GIT/dataframes_by_chain/sym/training_sn1_ft_11pt.csv")
df_path_sn2_11pt = os.path.join(main_dir, "heavy_data_NO_GIT/dataframes_by_chain/sym/training_sn2_ft_11pt.csv")

df_path_sn1_13pt = os.path.join(main_dir, "heavy_data_NO_GIT/dataframes_by_chain/sym/training_sn1_ft_13pt.csv")
df_path_sn2_13pt = os.path.join(main_dir, "heavy_data_NO_GIT/dataframes_by_chain/sym/training_sn2_ft_13pt.csv")

df_path_sn1_15pt = os.path.join(main_dir, "heavy_data_NO_GIT/dataframes_by_chain/sym/training_sn1_ft_15pt.csv")
df_path_sn2_15pt = os.path.join(main_dir, "heavy_data_NO_GIT/dataframes_by_chain/sym/training_sn2_ft_15pt.csv")

df_path_sn1_17pt = os.path.join(main_dir, "heavy_data_NO_GIT/dataframes_by_chain/sym/training_sn1_ft_17pt.csv")
df_path_sn2_17pt = os.path.join(main_dir, "heavy_data_NO_GIT/dataframes_by_chain/sym/training_sn2_ft_17pt.csv")

df_sn1_3pt = pd.read_csv(df_path_sn1_3pt)
df_sn2_3pt = pd.read_csv(df_path_sn2_3pt)

df_sn1_5pt = pd.read_csv(df_path_sn1_5pt)
df_sn2_5pt = pd.read_csv(df_path_sn2_5pt)

df_sn1_7pt = pd.read_csv(df_path_sn1_7pt)
df_sn2_7pt = pd.read_csv(df_path_sn2_7pt)

df_sn1_9pt = pd.read_csv(df_path_sn1_9pt)
df_sn2_9pt = pd.read_csv(df_path_sn2_9pt)

df_sn1_11pt = pd.read_csv(df_path_sn1_11pt)
df_sn2_11pt = pd.read_csv(df_path_sn2_11pt)

df_sn1_13pt = pd.read_csv(df_path_sn1_13pt)
df_sn2_13pt = pd.read_csv(df_path_sn2_13pt)

df_sn1_15pt = pd.read_csv(df_path_sn1_15pt)
df_sn2_15pt = pd.read_csv(df_path_sn2_15pt)

df_sn1_17pt = pd.read_csv(df_path_sn1_17pt)
df_sn2_17pt = pd.read_csv(df_path_sn2_17pt)

df_sn1_3pt.drop('Unnamed: 0', axis=1, inplace=True)
df_sn2_3pt.drop('Unnamed: 0', axis=1, inplace=True)

df_sn1_5pt.drop('Unnamed: 0', axis=1, inplace=True)
df_sn2_5pt.drop('Unnamed: 0', axis=1, inplace=True)

df_sn1_7pt.drop('Unnamed: 0', axis=1, inplace=True)
df_sn2_7pt.drop('Unnamed: 0', axis=1, inplace=True)

df_sn1_9pt.drop('Unnamed: 0', axis=1, inplace=True)
df_sn2_9pt.drop('Unnamed: 0', axis=1, inplace=True)

df_sn1_11pt.drop('Unnamed: 0', axis=1, inplace=True)
df_sn2_11pt.drop('Unnamed: 0', axis=1, inplace=True)

df_sn1_13pt.drop('Unnamed: 0', axis=1, inplace=True)
df_sn2_13pt.drop('Unnamed: 0', axis=1, inplace=True)

df_sn1_15pt.drop('Unnamed: 0', axis=1, inplace=True)
df_sn2_15pt.drop('Unnamed: 0', axis=1, inplace=True)

df_sn1_17pt.drop('Unnamed: 0', axis=1, inplace=True)
df_sn2_17pt.drop('Unnamed: 0', axis=1, inplace=True)

## Basic training parameters

In [ ]:
random_state = 76
results_summary = []

def _pred_from_proba(model, X, cutoff=0.5):
    """Return hard predictions using predict_proba (preferred) or predict()."""
    if hasattr(model, 'predict_proba'):
        proba = model.predict_proba(X)
        # assume binary classifier: take prob of class 1
        if proba.ndim == 2 and proba.shape[1] >= 2:
            p1 = proba[:, 1]
        else:
            p1 = np.ravel(proba)
        return (p1 >= cutoff).astype(int)
    return model.predict(X)

test_size = 0.30
sampling_fraction = 0.1  # kept for consistency with previous code (not used here)

# -------------------------------------------------------------------
# sn1 - 3 pt
# -------------------------------------------------------------------
print("\n==================== sn1 3 pt ====================")
X_sn1_3pt = df_sn1_3pt.iloc[:, :-1]
y_sn1_3pt = df_sn1_3pt.iloc[:, -1]

X_sn1_3pt_train, X_sn1_3pt_test, y_sn1_3pt_train, y_sn1_3pt_test = train_test_split(
    X_sn1_3pt, y_sn1_3pt, test_size=test_size, shuffle=True, random_state=random_state
)

scaler_sn1_3pt = StandardScaler()
X_sn1_3pt_train_scaled = scaler_sn1_3pt.fit_transform(X_sn1_3pt_train)
X_sn1_3pt_test_scaled = scaler_sn1_3pt.transform(X_sn1_3pt_test)

log_reg_sn1_3pt = LogisticMomentumGS(
        n_features=X_sn1_3pt_train_scaled.shape[1],
        use_scaler=False,          # we already scaled above
        n_splits=5,
        random_state=random_state,
        scoring="accuracy",
        n_jobs=1,
        verbose=0
    )

log_reg_sn1_3pt.fit(X_sn1_3pt_train_scaled, y_sn1_3pt_train)

log_reg_sn1_3pt_cm = prob_cutoff_confusion_matrix(log_reg_sn1_3pt, X_sn1_3pt_test_scaled, y_sn1_3pt_test)
log_reg_sn1_3pt_acc = np.trace(log_reg_sn1_3pt_cm) / len(y_sn1_3pt_test)

print("log_reg sn1 3pt accuracy:", log_reg_sn1_3pt_acc)
print("log_reg sn1 3pt confusion matrix:\n", log_reg_sn1_3pt_cm)


log_reg_sn1_3pt_y_pred = _pred_from_proba(log_reg_sn1_3pt, X_sn1_3pt_test_scaled, cutoff=0.5)
log_reg_sn1_3pt_f1 = f1_score(y_sn1_3pt_test, log_reg_sn1_3pt_y_pred)
print("log_reg sn1_3pt f1:", log_reg_sn1_3pt_f1)
results_summary.append({
    'tag': 'sn1_3pt',
    'accuracy': float(log_reg_sn1_3pt_acc),
    'f1': float(log_reg_sn1_3pt_f1),
    'confusion_matrix': log_reg_sn1_3pt_cm.tolist(),
})

filename_model_sn1_3pt = os.path.join(save_dir, "log_reg_sn1_3pt_model_ft_last.sav")
filename_scaler_sn1_3pt = os.path.join(save_dir, "log_reg_sn1_3pt_scaler_ft_last.sav")

pickle.dump(log_reg_sn1_3pt, open(filename_model_sn1_3pt, "wb"))
pickle.dump(scaler_sn1_3pt, open(filename_scaler_sn1_3pt, "wb"))



# -------------------------------------------------------------------
# sn2 - 3 pt
# -------------------------------------------------------------------
print("\n==================== sn2 3 pt ====================")
X_sn2_3pt = df_sn2_3pt.iloc[:, :-1]
y_sn2_3pt = df_sn2_3pt.iloc[:, -1]

X_sn2_3pt_train, X_sn2_3pt_test, y_sn2_3pt_train, y_sn2_3pt_test = train_test_split(
    X_sn2_3pt, y_sn2_3pt, test_size=test_size, shuffle=True, random_state=random_state
)

scaler_sn2_3pt = StandardScaler()
X_sn2_3pt_train_scaled = scaler_sn2_3pt.fit_transform(X_sn2_3pt_train)
X_sn2_3pt_test_scaled = scaler_sn2_3pt.transform(X_sn2_3pt_test)

log_reg_sn2_3pt = LogisticMomentumGS(
        n_features=X_sn2_3pt_train_scaled.shape[1],
        use_scaler=False,          # we already scaled above
        n_splits=5,
        random_state=random_state,
        scoring="accuracy",
        n_jobs=1,
        verbose=0
    )

log_reg_sn2_3pt.fit(X_sn2_3pt_train_scaled, y_sn2_3pt_train)

log_reg_sn2_3pt_cm = prob_cutoff_confusion_matrix(log_reg_sn2_3pt, X_sn2_3pt_test_scaled, y_sn2_3pt_test)
log_reg_sn2_3pt_acc = np.trace(log_reg_sn2_3pt_cm) / len(y_sn2_3pt_test)

print("log_reg sn2 3pt accuracy:", log_reg_sn2_3pt_acc)
print("log_reg sn2 3pt confusion matrix:\n", log_reg_sn2_3pt_cm)


log_reg_sn2_3pt_y_pred = _pred_from_proba(log_reg_sn2_3pt, X_sn2_3pt_test_scaled, cutoff=0.5)
log_reg_sn2_3pt_f1 = f1_score(y_sn2_3pt_test, log_reg_sn2_3pt_y_pred)
print("log_reg sn2_3pt f1:", log_reg_sn2_3pt_f1)
results_summary.append({
    'tag': 'sn2_3pt',
    'accuracy': float(log_reg_sn2_3pt_acc),
    'f1': float(log_reg_sn2_3pt_f1),
    'confusion_matrix': log_reg_sn2_3pt_cm.tolist(),
})

filename_model_sn2_3pt = os.path.join(save_dir, "log_reg_sn2_3pt_model_ft_last.sav")
filename_scaler_sn2_3pt = os.path.join(save_dir, "log_reg_sn2_3pt_scaler_ft_last.sav")

pickle.dump(log_reg_sn2_3pt, open(filename_model_sn2_3pt, "wb"))
pickle.dump(scaler_sn2_3pt, open(filename_scaler_sn2_3pt, "wb"))

test_sn1_3pt = (X_sn1_3pt_test , y_sn1_3pt_test)

with open(os.path.join(data_arrays , "log_reg_test_sn1_3pt.pkl") , "wb") as f:
    pickle.dump(test_sn1_3pt, f)

test_sn2_3pt = (X_sn2_3pt_test , y_sn2_3pt_test)

with open(os.path.join(data_arrays , "log_reg_test_sn2_3pt.pkl") , "wb") as f:
    pickle.dump(test_sn2_3pt, f)

filename_nn_model_sn1_3pt = os.path.join(main_dir , "heavy_data_NO_GIT/models_scalers",'log_reg_sn1_3pt_model.sav')
filename_scaler_sn1_3pt = os.path.join(main_dir , "heavy_data_NO_GIT/models_scalers/",'log_reg_sn1_3pt_scaler.sav')
pickle.dump(log_reg_sn1_3pt, open(filename_nn_model_sn1_3pt, 'wb'))
pickle.dump(scaler_sn1_3pt, open(filename_scaler_sn1_3pt, 'wb'))

filename_nn_model_sn2_3pt = os.path.join(main_dir , "heavy_data_NO_GIT/models_scalers",'log_reg_sn2_3pt_model.sav')
filename_scaler_sn2_3pt = os.path.join(main_dir , "heavy_data_NO_GIT/models_scalers/",'log_reg_sn2_3pt_scaler.sav')
pickle.dump(log_reg_sn2_3pt, open(filename_nn_model_sn2_3pt, 'wb'))
pickle.dump(scaler_sn2_3pt, open(filename_scaler_sn2_3pt, 'wb'))
'''

## sn1 - 5 pt

In [ ]:
print("\n==================== sn1 5 pt ====================")
X_sn1_5pt = df_sn1_5pt.iloc[:, :-1]
y_sn1_5pt = df_sn1_5pt.iloc[:, -1]

X_sn1_5pt_train, X_sn1_5pt_test, y_sn1_5pt_train, y_sn1_5pt_test = train_test_split(
    X_sn1_5pt, y_sn1_5pt, test_size=test_size, shuffle=True, random_state=random_state
)

scaler_sn1_5pt = StandardScaler()
X_sn1_5pt_train_scaled = scaler_sn1_5pt.fit_transform(X_sn1_5pt_train)
X_sn1_5pt_test_scaled = scaler_sn1_5pt.transform(X_sn1_5pt_test)

log_reg_sn1_5pt = LogisticMomentumGS(
        n_features=X_sn1_5pt_train_scaled.shape[1],
        use_scaler=False,          # we already scaled above
        n_splits=5,
        random_state=random_state,
        scoring="accuracy",
        n_jobs=1,
        verbose=0
    )

log_reg_sn1_5pt.fit(X_sn1_5pt_train_scaled, y_sn1_5pt_train)

log_reg_sn1_5pt_cm = prob_cutoff_confusion_matrix(log_reg_sn1_5pt, X_sn1_5pt_test_scaled, y_sn1_5pt_test)
log_reg_sn1_5pt_acc = np.trace(log_reg_sn1_5pt_cm) / len(y_sn1_5pt_test)

print("log_reg sn1 5pt accuracy:", log_reg_sn1_5pt_acc)
print("log_reg sn1 5pt confusion matrix:\n", log_reg_sn1_5pt_cm)


log_reg_sn1_5pt_y_pred = _pred_from_proba(log_reg_sn1_5pt, X_sn1_5pt_test_scaled, cutoff=0.5)
log_reg_sn1_5pt_f1 = f1_score(y_sn1_5pt_test, log_reg_sn1_5pt_y_pred)
print("log_reg sn1_5pt f1:", log_reg_sn1_5pt_f1)
results_summary.append({
    'tag': 'sn1_5pt',
    'accuracy': float(log_reg_sn1_5pt_acc),
    'f1': float(log_reg_sn1_5pt_f1),
    'confusion_matrix': log_reg_sn1_5pt_cm.tolist(),
})

filename_model_sn1_5pt = os.path.join(save_dir, "log_reg_sn1_5pt_model_ft_last.sav")
filename_scaler_sn1_5pt = os.path.join(save_dir, "log_reg_sn1_5pt_scaler_ft_last.sav")

pickle.dump(log_reg_sn1_5pt, open(filename_model_sn1_5pt, "wb"))
pickle.dump(scaler_sn1_5pt, open(filename_scaler_sn1_5pt, "wb"))

## sn2 - 5 pt

In [ ]:
print("\n==================== sn2 5 pt ====================")
X_sn2_5pt = df_sn2_5pt.iloc[:, :-1]
y_sn2_5pt = df_sn2_5pt.iloc[:, -1]

X_sn2_5pt_train, X_sn2_5pt_test, y_sn2_5pt_train, y_sn2_5pt_test = train_test_split(
    X_sn2_5pt, y_sn2_5pt, test_size=test_size, shuffle=True, random_state=random_state
)

scaler_sn2_5pt = StandardScaler()
X_sn2_5pt_train_scaled = scaler_sn2_5pt.fit_transform(X_sn2_5pt_train)
X_sn2_5pt_test_scaled = scaler_sn2_5pt.transform(X_sn2_5pt_test)

log_reg_sn2_5pt = LogisticMomentumGS(
        n_features=X_sn2_5pt_train_scaled.shape[1],
        use_scaler=False,          # we already scaled above
        n_splits=5,
        random_state=random_state,
        scoring="accuracy",
        n_jobs=1,
        verbose=0
    )

log_reg_sn2_5pt.fit(X_sn2_5pt_train_scaled, y_sn2_5pt_train)

log_reg_sn2_5pt_cm = prob_cutoff_confusion_matrix(log_reg_sn2_5pt, X_sn2_5pt_test_scaled, y_sn2_5pt_test)
log_reg_sn2_5pt_acc = np.trace(log_reg_sn2_5pt_cm) / len(y_sn2_5pt_test)

print("log_reg sn2 5pt accuracy:", log_reg_sn2_5pt_acc)
print("log_reg sn2 5pt confusion matrix:\n", log_reg_sn2_5pt_cm)


log_reg_sn2_5pt_y_pred = _pred_from_proba(log_reg_sn2_5pt, X_sn2_5pt_test_scaled, cutoff=0.5)
log_reg_sn2_5pt_f1 = f1_score(y_sn2_5pt_test, log_reg_sn2_5pt_y_pred)
print("log_reg sn2_5pt f1:", log_reg_sn2_5pt_f1)
results_summary.append({
    'tag': 'sn2_5pt',
    'accuracy': float(log_reg_sn2_5pt_acc),
    'f1': float(log_reg_sn2_5pt_f1),
    'confusion_matrix': log_reg_sn2_5pt_cm.tolist(),
})

filename_model_sn2_5pt = os.path.join(save_dir, "log_reg_sn2_5pt_model_ft_last.sav")
filename_scaler_sn2_5pt = os.path.join(save_dir, "log_reg_sn2_5pt_scaler_ft_last.sav")

pickle.dump(log_reg_sn2_5pt, open(filename_model_sn2_5pt, "wb"))
pickle.dump(scaler_sn2_5pt, open(filename_scaler_sn2_5pt, "wb"))

test_sn1_5pt = (X_sn1_5pt_test , y_sn1_5pt_test)

with open(os.path.join(data_arrays , log_reg_sn1_5pt.pkl") , "wb") as f:
    pickle.dump(test_sn1_5pt, f)

test_sn2_5pt = (X_sn2_5pt_test , y_sn2_5pt_test)

with open(os.path.join(data_arrays , "log_reg_test_sn2_5pt.pkl") , "wb") as f:
    pickle.dump(test_sn2_5pt, f)

filename_nn_model_sn1_5pt = os.path.join(main_dir , "heavy_data_NO_GIT/models_scalers",'log_reg_sn1_5pt_model.sav')
filename_scaler_sn1_5pt = os.path.join(main_dir , "heavy_data_NO_GIT/models_scalers/",'log_reg_sn1_5pt_scaler.sav')
pickle.dump(log_reg_sn1_5pt, open(filename_nn_model_sn1_5pt, 'wb'))
pickle.dump(scaler_sn1_5pt, open(filename_scaler_sn1_5pt, 'wb'))

filename_nn_model_sn2_5pt = os.path.join(main_dir , "heavy_data_NO_GIT/models_scalers",'log_reg_sn2_5pt_model.sav')
filename_scaler_sn2_5pt = os.path.join(main_dir , "heavy_data_NO_GIT/models_scalers/",'log_reg_sn2_5pt_scaler.sav')
pickle.dump(log_reg_sn2_5pt, open(filename_nn_model_sn2_5pt, 'wb'))
pickle.dump(scaler_sn2_5pt, open(filename_scaler_sn2_5pt, 'wb'))

## sn1 - 7 pt

In [ ]:
print("\n==================== sn1 7 pt ====================")
X_sn1_7pt = df_sn1_7pt.iloc[:, :-1]
y_sn1_7pt = df_sn1_7pt.iloc[:, -1]

X_sn1_7pt_train, X_sn1_7pt_test, y_sn1_7pt_train, y_sn1_7pt_test = train_test_split(
    X_sn1_7pt, y_sn1_7pt, test_size=test_size, shuffle=True, random_state=random_state
)

scaler_sn1_7pt = StandardScaler()
X_sn1_7pt_train_scaled = scaler_sn1_7pt.fit_transform(X_sn1_7pt_train)
X_sn1_7pt_test_scaled = scaler_sn1_7pt.transform(X_sn1_7pt_test)

log_reg_sn1_7pt = LogisticMomentumGS(
        n_features=X_sn1_7pt_train_scaled.shape[1],
        use_scaler=False,          # we already scaled above
        n_splits=5,
        random_state=random_state,
        scoring="accuracy",
        n_jobs=1,
        verbose=0
    )

log_reg_sn1_7pt.fit(X_sn1_7pt_train_scaled, y_sn1_7pt_train)

log_reg_sn1_7pt_cm = prob_cutoff_confusion_matrix(log_reg_sn1_7pt, X_sn1_7pt_test_scaled, y_sn1_7pt_test)
log_reg_sn1_7pt_acc = np.trace(log_reg_sn1_7pt_cm) / len(y_sn1_7pt_test)

print("log_reg sn1 7pt accuracy:", log_reg_sn1_7pt_acc)
print("log_reg sn1 7pt confusion matrix:\n", log_reg_sn1_7pt_cm)


log_reg_sn1_7pt_y_pred = _pred_from_proba(log_reg_sn1_7pt, X_sn1_7pt_test_scaled, cutoff=0.5)
log_reg_sn1_7pt_f1 = f1_score(y_sn1_7pt_test, log_reg_sn1_7pt_y_pred)
print("log_reg sn1_7pt f1:", log_reg_sn1_7pt_f1)
results_summary.append({
    'tag': 'sn1_7pt',
    'accuracy': float(log_reg_sn1_7pt_acc),
    'f1': float(log_reg_sn1_7pt_f1),
    'confusion_matrix': log_reg_sn1_7pt_cm.tolist(),
})

filename_model_sn1_7pt = os.path.join(save_dir, "log_reg_sn1_7pt_model_ft_last.sav")
filename_scaler_sn1_7pt = os.path.join(save_dir, "log_reg_sn1_7pt_scaler_ft_last.sav")

pickle.dump(log_reg_sn1_7pt, open(filename_model_sn1_7pt, "wb"))
pickle.dump(scaler_sn1_7pt, open(filename_scaler_sn1_7pt, "wb"))

## sn2 - 7 pt

In [ ]:
print("\n==================== sn2 7 pt ====================")
X_sn2_7pt = df_sn2_7pt.iloc[:, :-1]
y_sn2_7pt = df_sn2_7pt.iloc[:, -1]

X_sn2_7pt_train, X_sn2_7pt_test, y_sn2_7pt_train, y_sn2_7pt_test = train_test_split(
    X_sn2_7pt, y_sn2_7pt, test_size=test_size, shuffle=True, random_state=random_state
)

scaler_sn2_7pt = StandardScaler()
X_sn2_7pt_train_scaled = scaler_sn2_7pt.fit_transform(X_sn2_7pt_train)
X_sn2_7pt_test_scaled = scaler_sn2_7pt.transform(X_sn2_7pt_test)

log_reg_sn2_7pt = LogisticMomentumGS(
        n_features=X_sn2_7pt_train_scaled.shape[1],
        use_scaler=False,          # we already scaled above
        n_splits=5,
        random_state=random_state,
        scoring="accuracy",
        n_jobs=1,
        verbose=0
    )

log_reg_sn2_7pt.fit(X_sn2_7pt_train_scaled, y_sn2_7pt_train)

log_reg_sn2_7pt_cm = prob_cutoff_confusion_matrix(log_reg_sn2_7pt, X_sn2_7pt_test_scaled, y_sn2_7pt_test)
log_reg_sn2_7pt_acc = np.trace(log_reg_sn2_7pt_cm) / len(y_sn2_7pt_test)

print("log_reg sn2 7pt accuracy:", log_reg_sn2_7pt_acc)
print("log_reg sn2 7pt confusion matrix:\n", log_reg_sn2_7pt_cm)


log_reg_sn2_7pt_y_pred = _pred_from_proba(log_reg_sn2_7pt, X_sn2_7pt_test_scaled, cutoff=0.5)
log_reg_sn2_7pt_f1 = f1_score(y_sn2_7pt_test, log_reg_sn2_7pt_y_pred)
print("log_reg sn2_7pt f1:", log_reg_sn2_7pt_f1)
results_summary.append({
    'tag': 'sn2_7pt',
    'accuracy': float(log_reg_sn2_7pt_acc),
    'f1': float(log_reg_sn2_7pt_f1),
    'confusion_matrix': log_reg_sn2_7pt_cm.tolist(),
})

filename_model_sn2_7pt = os.path.join(save_dir, "log_reg_sn2_7pt_model_ft_last.sav")
filename_scaler_sn2_7pt = os.path.join(save_dir, "log_reg_sn2_7pt_scaler_ft_last.sav")

pickle.dump(log_reg_sn2_7pt, open(filename_model_sn2_7pt, "wb"))
pickle.dump(scaler_sn2_7pt, open(filename_scaler_sn2_7pt, "wb"))

test_sn1_7pt = (X_sn1_7pt_test , y_sn1_7pt_test)

with open(os.path.join(data_arrays , log_reg_sn1_7pt.pkl") , "wb") as f:
    pickle.dump(test_sn1_7pt, f)

test_sn2_7pt = (X_sn2_7pt_test , y_sn2_7pt_test)

with open(os.path.join(data_arrays , "log_reg_test_sn2_7pt.pkl") , "wb") as f:
    pickle.dump(test_sn2_7pt, f)

filename_nn_model_sn1_7pt = os.path.join(main_dir , "heavy_data_NO_GIT/models_scalers",'log_reg_sn1_7pt_model.sav')
filename_scaler_sn1_7pt = os.path.join(main_dir , "heavy_data_NO_GIT/models_scalers/",'log_reg_sn1_7pt_scaler.sav')
pickle.dump(log_reg_sn1_7pt, open(filename_nn_model_sn1_7pt, 'wb'))
pickle.dump(scaler_sn1_7pt, open(filename_scaler_sn1_7pt, 'wb'))

filename_nn_model_sn2_7pt = os.path.join(main_dir , "heavy_data_NO_GIT/models_scalers",'log_reg_sn2_7pt_model.sav')
filename_scaler_sn2_7pt = os.path.join(main_dir , "heavy_data_NO_GIT/models_scalers/",'log_reg_sn2_7pt_scaler.sav')
pickle.dump(log_reg_sn2_7pt, open(filename_nn_model_sn2_7pt, 'wb'))
pickle.dump(scaler_sn2_7pt, open(filename_scaler_sn2_7pt, 'wb'))

## sn1 - 9 pt

In [ ]:
print("\n==================== sn1 9 pt ====================")
X_sn1_9pt = df_sn1_9pt.iloc[:, :-1]
y_sn1_9pt = df_sn1_9pt.iloc[:, -1]

X_sn1_9pt_train, X_sn1_9pt_test, y_sn1_9pt_train, y_sn1_9pt_test = train_test_split(
    X_sn1_9pt, y_sn1_9pt, test_size=test_size, shuffle=True, random_state=random_state
)

scaler_sn1_9pt = StandardScaler()
X_sn1_9pt_train_scaled = scaler_sn1_9pt.fit_transform(X_sn1_9pt_train)
X_sn1_9pt_test_scaled = scaler_sn1_9pt.transform(X_sn1_9pt_test)

log_reg_sn1_9pt = LogisticMomentumGS(
        n_features=X_sn1_9pt_train_scaled.shape[1],
        use_scaler=False,          # we already scaled above
        n_splits=5,
        random_state=random_state,
        scoring="accuracy",
        n_jobs=1,
        verbose=0
    )

log_reg_sn1_9pt.fit(X_sn1_9pt_train_scaled, y_sn1_9pt_train)

log_reg_sn1_9pt_cm = prob_cutoff_confusion_matrix(log_reg_sn1_9pt, X_sn1_9pt_test_scaled, y_sn1_9pt_test)
log_reg_sn1_9pt_acc = np.trace(log_reg_sn1_9pt_cm) / len(y_sn1_9pt_test)

print("log_reg sn1 9pt accuracy:", log_reg_sn1_9pt_acc)
print("log_reg sn1 9pt confusion matrix:\n", log_reg_sn1_9pt_cm)


log_reg_sn1_9pt_y_pred = _pred_from_proba(log_reg_sn1_9pt, X_sn1_9pt_test_scaled, cutoff=0.5)
log_reg_sn1_9pt_f1 = f1_score(y_sn1_9pt_test, log_reg_sn1_9pt_y_pred)
print("log_reg sn1_9pt f1:", log_reg_sn1_9pt_f1)
results_summary.append({
    'tag': 'sn1_9pt',
    'accuracy': float(log_reg_sn1_9pt_acc),
    'f1': float(log_reg_sn1_9pt_f1),
    'confusion_matrix': log_reg_sn1_9pt_cm.tolist(),
})

filename_model_sn1_9pt = os.path.join(save_dir, "log_reg_sn1_9pt_model_ft_last.sav")
filename_scaler_sn1_9pt = os.path.join(save_dir, "log_reg_sn1_9pt_scaler_ft_last.sav")

pickle.dump(log_reg_sn1_9pt, open(filename_model_sn1_9pt, "wb"))
pickle.dump(scaler_sn1_9pt, open(filename_scaler_sn1_9pt, "wb"))

## sn2 - 9 pt

In [ ]:
print("\n==================== sn2 9 pt ====================")
X_sn2_9pt = df_sn2_9pt.iloc[:, :-1]
y_sn2_9pt = df_sn2_9pt.iloc[:, -1]

X_sn2_9pt_train, X_sn2_9pt_test, y_sn2_9pt_train, y_sn2_9pt_test = train_test_split(
    X_sn2_9pt, y_sn2_9pt, test_size=test_size, shuffle=True, random_state=random_state
)

scaler_sn2_9pt = StandardScaler()
X_sn2_9pt_train_scaled = scaler_sn2_9pt.fit_transform(X_sn2_9pt_train)
X_sn2_9pt_test_scaled = scaler_sn2_9pt.transform(X_sn2_9pt_test)

log_reg_sn2_9pt = LogisticMomentumGS(
        n_features=X_sn2_9pt_train_scaled.shape[1],
        use_scaler=False,          # we already scaled above
        n_splits=5,
        random_state=random_state,
        scoring="accuracy",
        n_jobs=1,
        verbose=0
    )

log_reg_sn2_9pt.fit(X_sn2_9pt_train_scaled, y_sn2_9pt_train)

log_reg_sn2_9pt_cm = prob_cutoff_confusion_matrix(log_reg_sn2_9pt, X_sn2_9pt_test_scaled, y_sn2_9pt_test)
log_reg_sn2_9pt_acc = np.trace(log_reg_sn2_9pt_cm) / len(y_sn2_9pt_test)

print("log_reg sn2 9pt accuracy:", log_reg_sn2_9pt_acc)
print("log_reg sn2 9pt confusion matrix:\n", log_reg_sn2_9pt_cm)


log_reg_sn2_9pt_y_pred = _pred_from_proba(log_reg_sn2_9pt, X_sn2_9pt_test_scaled, cutoff=0.5)
log_reg_sn2_9pt_f1 = f1_score(y_sn2_9pt_test, log_reg_sn2_9pt_y_pred)
print("log_reg sn2_9pt f1:", log_reg_sn2_9pt_f1)
results_summary.append({
    'tag': 'sn2_9pt',
    'accuracy': float(log_reg_sn2_9pt_acc),
    'f1': float(log_reg_sn2_9pt_f1),
    'confusion_matrix': log_reg_sn2_9pt_cm.tolist(),
})

filename_model_sn2_9pt = os.path.join(save_dir, "log_reg_sn2_9pt_model_ft_last.sav")
filename_scaler_sn2_9pt = os.path.join(save_dir, "log_reg_sn2_9pt_scaler_ft_last.sav")

pickle.dump(log_reg_sn2_9pt, open(filename_model_sn2_9pt, "wb"))
pickle.dump(scaler_sn2_9pt, open(filename_scaler_sn2_9pt, "wb"))

test_sn1_9pt = (X_sn1_9pt_test , y_sn1_9pt_test)

with open(os.path.join(data_arrays , log_reg_sn1_9pt.pkl") , "wb") as f:
    pickle.dump(test_sn1_9pt, f)

test_sn2_9pt = (X_sn2_9pt_test , y_sn2_9pt_test)

with open(os.path.join(data_arrays , "log_reg_test_sn2_9pt.pkl") , "wb") as f:
    pickle.dump(test_sn2_9pt, f)

filename_nn_model_sn1_9pt = os.path.join(main_dir , "heavy_data_NO_GIT/models_scalers",'log_reg_sn1_9pt_model.sav')
filename_scaler_sn1_9pt = os.path.join(main_dir , "heavy_data_NO_GIT/models_scalers/",'log_reg_sn1_9pt_scaler.sav')
pickle.dump(log_reg_sn1_9pt, open(filename_nn_model_sn1_9pt, 'wb'))
pickle.dump(scaler_sn1_9pt, open(filename_scaler_sn1_9pt, 'wb'))

filename_nn_model_sn2_9pt = os.path.join(main_dir , "heavy_data_NO_GIT/models_scalers",'log_reg_sn2_9pt_model.sav')
filename_scaler_sn2_9pt = os.path.join(main_dir , "heavy_data_NO_GIT/models_scalers/",'log_reg_sn2_9pt_scaler.sav')
pickle.dump(log_reg_sn2_9pt, open(filename_nn_model_sn2_9pt, 'wb'))
pickle.dump(scaler_sn2_9pt, open(filename_scaler_sn2_9pt, 'wb'))


# -------------------------------------------------------------------

#  11 pt

# -------------------------------------------------------------------

## sn1 - 11 pt

In [ ]:
print("\n==================== sn1 11 pt ====================")
X_sn1_11pt = df_sn1_11pt.iloc[:, :-1]
y_sn1_11pt = df_sn1_11pt.iloc[:, -1]

X_sn1_11pt_train, X_sn1_11pt_test, y_sn1_11pt_train, y_sn1_11pt_test = train_test_split(
    X_sn1_11pt, y_sn1_11pt, test_size=test_size, shuffle=True, random_state=random_state
)

scaler_sn1_11pt = StandardScaler()
X_sn1_11pt_train_scaled = scaler_sn1_11pt.fit_transform(X_sn1_11pt_train)
X_sn1_11pt_test_scaled = scaler_sn1_11pt.transform(X_sn1_11pt_test)

log_reg_sn1_11pt = LogisticMomentumGS(
        n_features=X_sn1_11pt_train_scaled.shape[1],
        use_scaler=False,          # we already scaled above
        n_splits=5,
        random_state=random_state,
        scoring="accuracy",
        n_jobs=1,
        verbose=0
    )

log_reg_sn1_11pt.fit(X_sn1_11pt_train_scaled, y_sn1_11pt_train)

log_reg_sn1_11pt_cm = prob_cutoff_confusion_matrix(log_reg_sn1_11pt, X_sn1_11pt_test_scaled, y_sn1_11pt_test)
log_reg_sn1_11pt_acc = np.trace(log_reg_sn1_11pt_cm) / len(y_sn1_11pt_test)

print("log_reg sn1 11pt accuracy:", log_reg_sn1_11pt_acc)
print("log_reg sn1 11pt confusion matrix:\n", log_reg_sn1_11pt_cm)


log_reg_sn1_11pt_y_pred = _pred_from_proba(log_reg_sn1_11pt, X_sn1_11pt_test_scaled, cutoff=0.5)
log_reg_sn1_11pt_f1 = f1_score(y_sn1_11pt_test, log_reg_sn1_11pt_y_pred)
print("log_reg sn1_11pt f1:", log_reg_sn1_11pt_f1)
results_summary.append({
    'tag': 'sn1_11pt',
    'accuracy': float(log_reg_sn1_11pt_acc),
    'f1': float(log_reg_sn1_11pt_f1),
    'confusion_matrix': log_reg_sn1_11pt_cm.tolist(),
})

filename_model_sn1_11pt = os.path.join(save_dir, "log_reg_sn1_11pt_model_ft_last.sav")
filename_scaler_sn1_11pt = os.path.join(save_dir, "log_reg_sn1_11pt_scaler_ft_last.sav")

pickle.dump(log_reg_sn1_11pt, open(filename_model_sn1_11pt, "wb"))
pickle.dump(scaler_sn1_11pt, open(filename_scaler_sn1_11pt, "wb"))

## sn2 - 11 pt

In [ ]:
print("\n==================== sn2 11 pt ====================")
X_sn2_11pt = df_sn2_11pt.iloc[:, :-1]
y_sn2_11pt = df_sn2_11pt.iloc[:, -1]

X_sn2_11pt_train, X_sn2_11pt_test, y_sn2_11pt_train, y_sn2_11pt_test = train_test_split(
    X_sn2_11pt, y_sn2_11pt, test_size=test_size, shuffle=True, random_state=random_state
)

scaler_sn2_11pt = StandardScaler()
X_sn2_11pt_train_scaled = scaler_sn2_11pt.fit_transform(X_sn2_11pt_train)
X_sn2_11pt_test_scaled = scaler_sn2_11pt.transform(X_sn2_11pt_test)

log_reg_sn2_11pt = LogisticMomentumGS(
        n_features=X_sn2_11pt_train_scaled.shape[1],
        use_scaler=False,          # we already scaled above
        n_splits=5,
        random_state=random_state,
        scoring="accuracy",
        n_jobs=1,
        verbose=0
    )

log_reg_sn2_11pt.fit(X_sn2_11pt_train_scaled, y_sn2_11pt_train)

log_reg_sn2_11pt_cm = prob_cutoff_confusion_matrix(log_reg_sn2_11pt, X_sn2_11pt_test_scaled, y_sn2_11pt_test)
log_reg_sn2_11pt_acc = np.trace(log_reg_sn2_11pt_cm) / len(y_sn2_11pt_test)

print("log_reg sn2 11pt accuracy:", log_reg_sn2_11pt_acc)
print("log_reg sn2 11pt confusion matrix:\n", log_reg_sn2_11pt_cm)


log_reg_sn2_11pt_y_pred = _pred_from_proba(log_reg_sn2_11pt, X_sn2_11pt_test_scaled, cutoff=0.5)
log_reg_sn2_11pt_f1 = f1_score(y_sn2_11pt_test, log_reg_sn2_11pt_y_pred)
print("log_reg sn2_11pt f1:", log_reg_sn2_11pt_f1)
results_summary.append({
    'tag': 'sn2_11pt',
    'accuracy': float(log_reg_sn2_11pt_acc),
    'f1': float(log_reg_sn2_11pt_f1),
    'confusion_matrix': log_reg_sn2_11pt_cm.tolist(),
})

filename_model_sn2_11pt = os.path.join(save_dir, "log_reg_sn2_11pt_model_ft_last.sav")
filename_scaler_sn2_11pt = os.path.join(save_dir, "log_reg_sn2_11pt_scaler_ft_last.sav")

pickle.dump(log_reg_sn2_11pt, open(filename_model_sn2_11pt, "wb"))
pickle.dump(scaler_sn2_11pt, open(filename_scaler_sn2_11pt, "wb"))

test_sn1_11pt = (X_sn1_11pt_test , y_sn1_11pt_test)

with open(os.path.join(data_arrays , log_reg_sn1_11pt.pkl") , "wb") as f:
    pickle.dump(test_sn1_11pt, f)

test_sn2_11pt = (X_sn2_11pt_test , y_sn2_11pt_test)

with open(os.path.join(data_arrays , "log_reg_test_sn2_11pt.pkl") , "wb") as f:
    pickle.dump(test_sn2_11pt, f)

filename_nn_model_sn1_11pt = os.path.join(main_dir , "heavy_data_NO_GIT/models_scalers",'log_reg_sn1_11pt_model.sav')
filename_scaler_sn1_11pt = os.path.join(main_dir , "heavy_data_NO_GIT/models_scalers/",'log_reg_sn1_11pt_scaler.sav')
pickle.dump(log_reg_sn1_11pt, open(filename_nn_model_sn1_11pt, 'wb'))
pickle.dump(scaler_sn1_11pt, open(filename_scaler_sn1_11pt, 'wb'))

filename_nn_model_sn2_11pt = os.path.join(main_dir , "heavy_data_NO_GIT/models_scalers",'log_reg_sn2_11pt_model.sav')
filename_scaler_sn2_11pt = os.path.join(main_dir , "heavy_data_NO_GIT/models_scalers/",'log_reg_sn2_11pt_scaler.sav')
pickle.dump(log_reg_sn2_11pt, open(filename_nn_model_sn2_11pt, 'wb'))
pickle.dump(scaler_sn2_11pt, open(filename_scaler_sn2_11pt, 'wb'))

# -------------------------------------------------------------------

#  13 pt

# -------------------------------------------------------------------

## sn1 - 13 pt

In [ ]:
print("\n==================== sn1 13 pt ====================")
X_sn1_13pt = df_sn1_13pt.iloc[:, :-1]
y_sn1_13pt = df_sn1_13pt.iloc[:, -1]

X_sn1_13pt_train, X_sn1_13pt_test, y_sn1_13pt_train, y_sn1_13pt_test = train_test_split(
    X_sn1_13pt, y_sn1_13pt, test_size=test_size, shuffle=True, random_state=random_state
)

scaler_sn1_13pt = StandardScaler()
X_sn1_13pt_train_scaled = scaler_sn1_13pt.fit_transform(X_sn1_13pt_train)
X_sn1_13pt_test_scaled = scaler_sn1_13pt.transform(X_sn1_13pt_test)

log_reg_sn1_13pt = LogisticMomentumGS(
        n_features=X_sn1_13pt_train_scaled.shape[1],
        use_scaler=False,          # we already scaled above
        n_splits=5,
        random_state=random_state,
        scoring="accuracy",
        n_jobs=1,
        verbose=0
    )

log_reg_sn1_13pt.fit(X_sn1_13pt_train_scaled, y_sn1_13pt_train)

log_reg_sn1_13pt_cm = prob_cutoff_confusion_matrix(log_reg_sn1_13pt, X_sn1_13pt_test_scaled, y_sn1_13pt_test)
log_reg_sn1_13pt_acc = np.trace(log_reg_sn1_13pt_cm) / len(y_sn1_13pt_test)

print("log_reg sn1 13pt accuracy:", log_reg_sn1_13pt_acc)
print("log_reg sn1 13pt confusion matrix:\n", log_reg_sn1_13pt_cm)


log_reg_sn1_13pt_y_pred = _pred_from_proba(log_reg_sn1_13pt, X_sn1_13pt_test_scaled, cutoff=0.5)
log_reg_sn1_13pt_f1 = f1_score(y_sn1_13pt_test, log_reg_sn1_13pt_y_pred)
print("log_reg sn1_13pt f1:", log_reg_sn1_13pt_f1)
results_summary.append({
    'tag': 'sn1_13pt',
    'accuracy': float(log_reg_sn1_13pt_acc),
    'f1': float(log_reg_sn1_13pt_f1),
    'confusion_matrix': log_reg_sn1_13pt_cm.tolist(),
})

filename_model_sn1_13pt = os.path.join(save_dir, "log_reg_sn1_13pt_model_ft_last.sav")
filename_scaler_sn1_13pt = os.path.join(save_dir, "log_reg_sn1_13pt_scaler_ft_last.sav")

pickle.dump(log_reg_sn1_13pt, open(filename_model_sn1_13pt, "wb"))
pickle.dump(scaler_sn1_13pt, open(filename_scaler_sn1_13pt, "wb"))

## sn2 - 13 pt

In [ ]:
print("\n==================== sn2 13 pt ====================")
X_sn2_13pt = df_sn2_13pt.iloc[:, :-1]
y_sn2_13pt = df_sn2_13pt.iloc[:, -1]

X_sn2_13pt_train, X_sn2_13pt_test, y_sn2_13pt_train, y_sn2_13pt_test = train_test_split(
    X_sn2_13pt, y_sn2_13pt, test_size=test_size, shuffle=True, random_state=random_state
)

scaler_sn2_13pt = StandardScaler()
X_sn2_13pt_train_scaled = scaler_sn2_13pt.fit_transform(X_sn2_13pt_train)
X_sn2_13pt_test_scaled = scaler_sn2_13pt.transform(X_sn2_13pt_test)

log_reg_sn2_13pt = LogisticMomentumGS(
        n_features=X_sn2_13pt_train_scaled.shape[1],
        use_scaler=False,          # we already scaled above
        n_splits=5,
        random_state=random_state,
        scoring="accuracy",
        n_jobs=1,
        verbose=0
    )

log_reg_sn2_13pt.fit(X_sn2_13pt_train_scaled, y_sn2_13pt_train)

log_reg_sn2_13pt_cm = prob_cutoff_confusion_matrix(log_reg_sn2_13pt, X_sn2_13pt_test_scaled, y_sn2_13pt_test)
log_reg_sn2_13pt_acc = np.trace(log_reg_sn2_13pt_cm) / len(y_sn2_13pt_test)

print("log_reg sn2 13pt accuracy:", log_reg_sn2_13pt_acc)
print("log_reg sn2 13pt confusion matrix:\n", log_reg_sn2_13pt_cm)


log_reg_sn2_13pt_y_pred = _pred_from_proba(log_reg_sn2_13pt, X_sn2_13pt_test_scaled, cutoff=0.5)
log_reg_sn2_13pt_f1 = f1_score(y_sn2_13pt_test, log_reg_sn2_13pt_y_pred)
print("log_reg sn2_13pt f1:", log_reg_sn2_13pt_f1)
results_summary.append({
    'tag': 'sn2_13pt',
    'accuracy': float(log_reg_sn2_13pt_acc),
    'f1': float(log_reg_sn2_13pt_f1),
    'confusion_matrix': log_reg_sn2_13pt_cm.tolist(),
})

filename_model_sn2_13pt = os.path.join(save_dir, "log_reg_sn2_13pt_model_ft_last.sav")
filename_scaler_sn2_13pt = os.path.join(save_dir, "log_reg_sn2_13pt_scaler_ft_last.sav")

pickle.dump(log_reg_sn2_13pt, open(filename_model_sn2_13pt, "wb"))
pickle.dump(scaler_sn2_13pt, open(filename_scaler_sn2_13pt, "wb"))

test_sn1_13pt = (X_sn1_13pt_test , y_sn1_13pt_test)

with open(os.path.join(data_arrays , log_reg_sn1_13pt.pkl") , "wb") as f:
    pickle.dump(test_sn1_13pt, f)

test_sn2_13pt = (X_sn2_13pt_test , y_sn2_13pt_test)

with open(os.path.join(data_arrays , "log_reg_test_sn2_13pt.pkl") , "wb") as f:
    pickle.dump(test_sn2_13pt, f)

filename_nn_model_sn1_13pt = os.path.join(main_dir , "heavy_data_NO_GIT/models_scalers",'log_reg_sn1_13pt_model.sav')
filename_scaler_sn1_13pt = os.path.join(main_dir , "heavy_data_NO_GIT/models_scalers/",'log_reg_sn1_13pt_scaler.sav')
pickle.dump(log_reg_sn1_13pt, open(filename_nn_model_sn1_13pt, 'wb'))
pickle.dump(scaler_sn1_13pt, open(filename_scaler_sn1_13pt, 'wb'))

filename_nn_model_sn2_13pt = os.path.join(main_dir , "heavy_data_NO_GIT/models_scalers",'log_reg_sn2_13pt_model.sav')
filename_scaler_sn2_13pt = os.path.join(main_dir , "heavy_data_NO_GIT/models_scalers/",'log_reg_sn2_13pt_scaler.sav')
pickle.dump(log_reg_sn2_13pt, open(filename_nn_model_sn2_13pt, 'wb'))
pickle.dump(scaler_sn2_13pt, open(filename_scaler_sn2_13pt, 'wb'))

# -------------------------------------------------------------------

#  15 pt

# -------------------------------------------------------------------

## sn1 - 15 pt

In [ ]:
print("\n==================== sn1 15 pt ====================")
X_sn1_15pt = df_sn1_15pt.iloc[:, :-1]
y_sn1_15pt = df_sn1_15pt.iloc[:, -1]

X_sn1_15pt_train, X_sn1_15pt_test, y_sn1_15pt_train, y_sn1_15pt_test = train_test_split(
    X_sn1_15pt, y_sn1_15pt, test_size=test_size, shuffle=True, random_state=random_state
)

scaler_sn1_15pt = StandardScaler()
X_sn1_15pt_train_scaled = scaler_sn1_15pt.fit_transform(X_sn1_15pt_train)
X_sn1_15pt_test_scaled = scaler_sn1_15pt.transform(X_sn1_15pt_test)

log_reg_sn1_15pt = LogisticMomentumGS(
        n_features=X_sn1_15pt_train_scaled.shape[1],
        use_scaler=False,          # we already scaled above
        n_splits=5,
        random_state=random_state,
        scoring="accuracy",
        n_jobs=1,
        verbose=0
    )

log_reg_sn1_15pt.fit(X_sn1_15pt_train_scaled, y_sn1_15pt_train)

log_reg_sn1_15pt_cm = prob_cutoff_confusion_matrix(log_reg_sn1_15pt, X_sn1_15pt_test_scaled, y_sn1_15pt_test)
log_reg_sn1_15pt_acc = np.trace(log_reg_sn1_15pt_cm) / len(y_sn1_15pt_test)

print("log_reg sn1 15pt accuracy:", log_reg_sn1_15pt_acc)
print("log_reg sn1 15pt confusion matrix:\n", log_reg_sn1_15pt_cm)


log_reg_sn1_15pt_y_pred = _pred_from_proba(log_reg_sn1_15pt, X_sn1_15pt_test_scaled, cutoff=0.5)
log_reg_sn1_15pt_f1 = f1_score(y_sn1_15pt_test, log_reg_sn1_15pt_y_pred)
print("log_reg sn1_15pt f1:", log_reg_sn1_15pt_f1)
results_summary.append({
    'tag': 'sn1_15pt',
    'accuracy': float(log_reg_sn1_15pt_acc),
    'f1': float(log_reg_sn1_15pt_f1),
    'confusion_matrix': log_reg_sn1_15pt_cm.tolist(),
})

filename_model_sn1_15pt = os.path.join(save_dir, "log_reg_sn1_15pt_model_ft_last.sav")
filename_scaler_sn1_15pt = os.path.join(save_dir, "log_reg_sn1_15pt_scaler_ft_last.sav")

pickle.dump(log_reg_sn1_15pt, open(filename_model_sn1_15pt, "wb"))
pickle.dump(scaler_sn1_15pt, open(filename_scaler_sn1_15pt, "wb"))

## sn2 - 15 pt

In [ ]:
print("\n==================== sn2 15 pt ====================")
X_sn2_15pt = df_sn2_15pt.iloc[:, :-1]
y_sn2_15pt = df_sn2_15pt.iloc[:, -1]

X_sn2_15pt_train, X_sn2_15pt_test, y_sn2_15pt_train, y_sn2_15pt_test = train_test_split(
    X_sn2_15pt, y_sn2_15pt, test_size=test_size, shuffle=True, random_state=random_state
)

scaler_sn2_15pt = StandardScaler()
X_sn2_15pt_train_scaled = scaler_sn2_15pt.fit_transform(X_sn2_15pt_train)
X_sn2_15pt_test_scaled = scaler_sn2_15pt.transform(X_sn2_15pt_test)

log_reg_sn2_15pt = LogisticMomentumGS(
        n_features=X_sn2_15pt_train_scaled.shape[1],
        use_scaler=False,          # we already scaled above
        n_splits=5,
        random_state=random_state,
        scoring="accuracy",
        n_jobs=1,
        verbose=0
    )

log_reg_sn2_15pt.fit(X_sn2_15pt_train_scaled, y_sn2_15pt_train)

log_reg_sn2_15pt_cm = prob_cutoff_confusion_matrix(log_reg_sn2_15pt, X_sn2_15pt_test_scaled, y_sn2_15pt_test)
log_reg_sn2_15pt_acc = np.trace(log_reg_sn2_15pt_cm) / len(y_sn2_15pt_test)

print("log_reg sn2 15pt accuracy:", log_reg_sn2_15pt_acc)
print("log_reg sn2 15pt confusion matrix:\n", log_reg_sn2_15pt_cm)


log_reg_sn2_15pt_y_pred = _pred_from_proba(log_reg_sn2_15pt, X_sn2_15pt_test_scaled, cutoff=0.5)
log_reg_sn2_15pt_f1 = f1_score(y_sn2_15pt_test, log_reg_sn2_15pt_y_pred)
print("log_reg sn2_15pt f1:", log_reg_sn2_15pt_f1)
results_summary.append({
    'tag': 'sn2_15pt',
    'accuracy': float(log_reg_sn2_15pt_acc),
    'f1': float(log_reg_sn2_15pt_f1),
    'confusion_matrix': log_reg_sn2_15pt_cm.tolist(),
})

filename_model_sn2_15pt = os.path.join(save_dir, "log_reg_sn2_15pt_model_ft_last.sav")
filename_scaler_sn2_15pt = os.path.join(save_dir, "log_reg_sn2_15pt_scaler_ft_last.sav")

pickle.dump(log_reg_sn2_15pt, open(filename_model_sn2_15pt, "wb"))
pickle.dump(scaler_sn2_15pt, open(filename_scaler_sn2_15pt, "wb"))

test_sn1_15pt = (X_sn1_15pt_test , y_sn1_15pt_test)

with open(os.path.join(data_arrays , log_reg_sn1_15pt.pkl") , "wb") as f:
    pickle.dump(test_sn1_15pt, f)

test_sn2_15pt = (X_sn2_15pt_test , y_sn2_15pt_test)

with open(os.path.join(data_arrays , "log_reg_test_sn2_15pt.pkl") , "wb") as f:
    pickle.dump(test_sn2_15pt, f)

filename_nn_model_sn1_15pt = os.path.join(main_dir , "heavy_data_NO_GIT/models_scalers",'log_reg_sn1_15pt_model.sav')
filename_scaler_sn1_15pt = os.path.join(main_dir , "heavy_data_NO_GIT/models_scalers/",'log_reg_sn1_15pt_scaler.sav')
pickle.dump(log_reg_sn1_15pt, open(filename_nn_model_sn1_15pt, 'wb'))
pickle.dump(scaler_sn1_15pt, open(filename_scaler_sn1_15pt, 'wb'))

filename_nn_model_sn2_15pt = os.path.join(main_dir , "heavy_data_NO_GIT/models_scalers",'log_reg_sn2_15pt_model.sav')
filename_scaler_sn2_15pt = os.path.join(main_dir , "heavy_data_NO_GIT/models_scalers/",'log_reg_sn2_15pt_scaler.sav')
pickle.dump(log_reg_sn2_15pt, open(filename_nn_model_sn2_15pt, 'wb'))
pickle.dump(scaler_sn2_15pt, open(filename_scaler_sn2_15pt, 'wb'))


# -------------------------------------------------------------------

#  17 pt

# -------------------------------------------------------------------

## sn1 - 17 pt

In [ ]:
print("\n==================== sn1 17 pt ====================")
X_sn1_17pt = df_sn1_17pt.iloc[:, :-1]
y_sn1_17pt = df_sn1_17pt.iloc[:, -1]

X_sn1_17pt_train, X_sn1_17pt_test, y_sn1_17pt_train, y_sn1_17pt_test = train_test_split(
    X_sn1_17pt, y_sn1_17pt, test_size=test_size, shuffle=True, random_state=random_state
)

scaler_sn1_17pt = StandardScaler()
X_sn1_17pt_train_scaled = scaler_sn1_17pt.fit_transform(X_sn1_17pt_train)
X_sn1_17pt_test_scaled = scaler_sn1_17pt.transform(X_sn1_17pt_test)

log_reg_sn1_17pt = LogisticMomentumGS(
        n_features=X_sn1_17pt_train_scaled.shape[1],
        use_scaler=False,          # we already scaled above
        n_splits=5,
        random_state=random_state,
        scoring="accuracy",
        n_jobs=1,
        verbose=0
    )

log_reg_sn1_17pt.fit(X_sn1_17pt_train_scaled, y_sn1_17pt_train)

log_reg_sn1_17pt_cm = prob_cutoff_confusion_matrix(log_reg_sn1_17pt, X_sn1_17pt_test_scaled, y_sn1_17pt_test)
log_reg_sn1_17pt_acc = np.trace(log_reg_sn1_17pt_cm) / len(y_sn1_17pt_test)

print("log_reg sn1 17pt accuracy:", log_reg_sn1_17pt_acc)
print("log_reg sn1 17pt confusion matrix:\n", log_reg_sn1_17pt_cm)


log_reg_sn1_17pt_y_pred = _pred_from_proba(log_reg_sn1_17pt, X_sn1_17pt_test_scaled, cutoff=0.5)
log_reg_sn1_17pt_f1 = f1_score(y_sn1_17pt_test, log_reg_sn1_17pt_y_pred)
print("log_reg sn1_17pt f1:", log_reg_sn1_17pt_f1)
results_summary.append({
    'tag': 'sn1_17pt',
    'accuracy': float(log_reg_sn1_17pt_acc),
    'f1': float(log_reg_sn1_17pt_f1),
    'confusion_matrix': log_reg_sn1_17pt_cm.tolist(),
})

filename_model_sn1_17pt = os.path.join(save_dir, "log_reg_sn1_17pt_model_ft_last.sav")
filename_scaler_sn1_17pt = os.path.join(save_dir, "log_reg_sn1_17pt_scaler_ft_last.sav")

pickle.dump(log_reg_sn1_17pt, open(filename_model_sn1_17pt, "wb"))
pickle.dump(scaler_sn1_17pt, open(filename_scaler_sn1_17pt, "wb"))

## sn2 - 17 pt

In [ ]:
print("\n==================== sn2 17 pt ====================")
X_sn2_17pt = df_sn2_17pt.iloc[:, :-1]
y_sn2_17pt = df_sn2_17pt.iloc[:, -1]

X_sn2_17pt_train, X_sn2_17pt_test, y_sn2_17pt_train, y_sn2_17pt_test = train_test_split(
    X_sn2_17pt, y_sn2_17pt, test_size=test_size, shuffle=True, random_state=random_state
)

scaler_sn2_17pt = StandardScaler()
X_sn2_17pt_train_scaled = scaler_sn2_17pt.fit_transform(X_sn2_17pt_train)
X_sn2_17pt_test_scaled = scaler_sn2_17pt.transform(X_sn2_17pt_test)

log_reg_sn2_17pt = LogisticMomentumGS(
        n_features=X_sn2_17pt_train_scaled.shape[1],
        use_scaler=False,          # we already scaled above
        n_splits=5,
        random_state=random_state,
        scoring="accuracy",
        n_jobs=1,
        verbose=0
    )

log_reg_sn2_17pt.fit(X_sn2_17pt_train_scaled, y_sn2_17pt_train)

log_reg_sn2_17pt_cm = prob_cutoff_confusion_matrix(log_reg_sn2_17pt, X_sn2_17pt_test_scaled, y_sn2_17pt_test)
log_reg_sn2_17pt_acc = np.trace(log_reg_sn2_17pt_cm) / len(y_sn2_17pt_test)

print("log_reg sn2 17pt accuracy:", log_reg_sn2_17pt_acc)
print("log_reg sn2 17pt confusion matrix:\n", log_reg_sn2_17pt_cm)


log_reg_sn2_17pt_y_pred = _pred_from_proba(log_reg_sn2_17pt, X_sn2_17pt_test_scaled, cutoff=0.5)
log_reg_sn2_17pt_f1 = f1_score(y_sn2_17pt_test, log_reg_sn2_17pt_y_pred)
print("log_reg sn2_17pt f1:", log_reg_sn2_17pt_f1)
results_summary.append({
    'tag': 'sn2_17pt',
    'accuracy': float(log_reg_sn2_17pt_acc),
    'f1': float(log_reg_sn2_17pt_f1),
    'confusion_matrix': log_reg_sn2_17pt_cm.tolist(),
})

filename_model_sn2_17pt = os.path.join(save_dir, "log_reg_sn2_17pt_model_ft_last.sav")
filename_scaler_sn2_17pt = os.path.join(save_dir, "log_reg_sn2_17pt_scaler_ft_last.sav")

pickle.dump(log_reg_sn2_17pt, open(filename_model_sn2_17pt, "wb"))
pickle.dump(scaler_sn2_17pt, open(filename_scaler_sn2_17pt, "wb"))

test_sn1_17pt = (X_sn1_17pt_test , y_sn1_17pt_test)

with open(os.path.join(data_arrays , log_reg_sn1_17pt.pkl") , "wb") as f:
    pickle.dump(test_sn1_17pt, f)

test_sn2_17pt = (X_sn2_17pt_test , y_sn2_17pt_test)

with open(os.path.join(data_arrays , "log_reg_test_sn2_17pt.pkl") , "wb") as f:
    pickle.dump(test_sn2_17pt, f)

filename_nn_model_sn1_17pt = os.path.join(main_dir , "heavy_data_NO_GIT/models_scalers",'log_reg_sn1_17pt_model.sav')
filename_scaler_sn1_17pt = os.path.join(main_dir , "heavy_data_NO_GIT/models_scalers/",'log_reg_sn1_17pt_scaler.sav')
pickle.dump(log_reg_sn1_17pt, open(filename_nn_model_sn1_17pt, 'wb'))
pickle.dump(scaler_sn1_17pt, open(filename_scaler_sn1_17pt, 'wb'))

filename_nn_model_sn2_17pt = os.path.join(main_dir , "heavy_data_NO_GIT/models_scalers",'log_reg_sn2_17pt_model.sav')
filename_scaler_sn2_17pt = os.path.join(main_dir , "heavy_data_NO_GIT/models_scalers/",'log_reg_sn2_17pt_scaler.sav')
pickle.dump(log_reg_sn2_17pt, open(filename_nn_model_sn2_17pt, 'wb'))
pickle.dump(scaler_sn2_17pt, open(filename_scaler_sn2_17pt, 'wb'))

## Save summary metrics

In [ ]:
results_txt_path = os.path.join(save_dir, "logreg_ft_last_sn1_sn2_metrics.txt")
with open(results_txt_path, "w", encoding="utf-8") as f:
    f.write("Logreg (FT_LAST) separate sn1/sn2 metrics\n")
    f.write(f"random_state={random_state}, test_size={test_size}\n\n")
    for r in results_summary:
        f.write(f"[{r['tag']}]\n")
        f.write(f"accuracy: {r['accuracy']:.6f}\n")
        f.write(f"f1: {r['f1']:.6f}\n")
        f.write("confusion_matrix:\n")
        cm = np.array(r['confusion_matrix'])
        f.write(str(cm) + "\n\n")

print("\nSaved metrics summary to:", results_txt_path)